# nnU-Net BraTS 2024 MEN-RT — Full Pipeline (Kaggle)

**Before running:**
1. GPU must be ON → Notebook Settings → Accelerator → **GPU T4 x2** or **P100**
2. Add your BraTS dataset as a Kaggle Dataset input
3. Add your GitHub token as a Kaggle Secret named `nnunet_kaggle`

**Pipeline overview:**
- Steps 1–2: Data preparation and nnU-Net preprocessing
- Step 3: K-fold cross-validation training (fold-by-fold)
- Step 4: Per-fold inference → segmentation predictions
- Steps 5–6: Evaluation and visualization

**Resuming an interrupted session:**
Re-run Cells 1–7, set `CONTINUE_TRAINING = True` in the relevant training cell, then re-run that cell.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, os, sys

try:
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(smi.stdout)
except FileNotFoundError:
    print('nvidia-smi not found — GPU is not enabled yet.\n')
    print('ACTION REQUIRED:')
    print('  1. Right side panel → Settings (pencil icon)')
    print('  2. Under Accelerator → select GPU T4 x2 or P100')
    print('  3. Click Save → kernel restarts automatically')
    print('  4. Re-run this cell — it should show GPU info')

import torch
cuda_ok = torch.cuda.is_available()
print(f'CUDA available : {cuda_ok}')
if cuda_ok:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB')
    print('\nGPU OK — safe to continue to Cell 2.')
else:
    raise RuntimeError(
        '\nNo GPU detected!\n'
        'Go to: right panel → Settings → Accelerator → GPU T4 x2\n'
        'Save, wait for kernel restart, then re-run this cell.'
    )

In [ ]:
# ── Cell 2: Locate BraTS dataset from Kaggle input ────────────────────────────
import glob, re
from collections import Counter
from pathlib import Path

KAGGLE_INPUT = '/kaggle/input'

# ── Known exact extracted paths (confirmed working structure) ─────────────────
_KNOWN_TRAIN = (
    f'{KAGGLE_INPUT}/datasets/maryampervaiz24029/brats-men-rt'
    '/BraTS2024-MEN-RT-TrainingData/BraTS-MEN-RT-Train-v2'
)
_KNOWN_VAL = (
    f'{KAGGLE_INPUT}/datasets/maryampervaiz24029/brats-men-rt'
    '/BraTS2024-MEN-RT-ValidationData/BraTS-MEN-RT-Val-v1'
)

def _find_zip(root: str, *names: str) -> str | None:
    for name in names:
        hits = sorted(glob.glob(f'{root}/**/{name}', recursive=True))
        if hits:
            return hits[0]
    return None

def _find_case_root(root: str) -> str | None:
    for pat in ('*_t1c.nii.gz', '*_t1c.nii', '*_t1ce.nii.gz', '*_t1ce.nii',
                '*_t1n.nii.gz', '*_t1n.nii', '*_t1.nii.gz', '*_t1.nii'):
        hits = sorted(glob.glob(f'{root}/**/{pat}', recursive=True))
        if hits:
            return str(Path(hits[0]).parent.parent)
    return None

def _scan_suffixes(root: str, max_files: int = 300) -> Counter:
    c: Counter = Counter()
    rex = re.compile(r'^.+?_([^_]+)\.nii(?:\.gz)?$', re.IGNORECASE)
    for pat in ('**/*.nii.gz', '**/*.nii'):
        for f in glob.glob(f'{root}/{pat}', recursive=True)[:max_files]:
            m = rex.match(Path(f).name)
            if m:
                c[m.group(1).lower()] += 1
    return c

# ── Priority 1: known extracted paths ────────────────────────────────────────
TRAIN_ZIP = _KNOWN_TRAIN if Path(_KNOWN_TRAIN).is_dir() else None
VAL_ZIP   = _KNOWN_VAL   if Path(_KNOWN_VAL).is_dir()   else None

# ── Priority 2: zip files ─────────────────────────────────────────────────────
if TRAIN_ZIP is None:
    TRAIN_ZIP = _find_zip(KAGGLE_INPUT,
        'BraTS2024-MEN-RT-TrainingData.zip', 'BraTS-MEN-RT-50cases.zip')
if VAL_ZIP is None:
    VAL_ZIP = _find_zip(KAGGLE_INPUT, 'BraTS2024-MEN-RT-ValidationData.zip')

# ── Priority 3: generic discovery ────────────────────────────────────────────
if TRAIN_ZIP is None:
    TRAIN_ZIP = _find_case_root(KAGGLE_INPUT)

print('Dataset discovery results:')
print(f'  Training source : {TRAIN_ZIP}')
print(f'  Validation src  : {VAL_ZIP}')

# ── Diagnostic ────────────────────────────────────────────────────────────────
suffix_counts: Counter = Counter()
if TRAIN_ZIP and not TRAIN_ZIP.endswith('.zip'):
    suffix_counts = _scan_suffixes(TRAIN_ZIP)
    print(f'\nNIfTI suffixes in training root:')
    for suf, n in sorted(suffix_counts.items(), key=lambda x: -x[1]):
        tag = ' <- IMAGE' if suf in ('t1c','t1ce','t1n','t1','flair') else (' <- LABEL' if suf == 'gtv' else '')
        print(f'  _{suf} : {n:4d} files{tag}')
    n_cases = sum(1 for p in Path(TRAIN_ZIP).iterdir() if p.is_dir())
    print(f'  Case folders : {n_cases}')
elif TRAIN_ZIP and TRAIN_ZIP.endswith('.zip'):
    print('  Source is a zip — extracted during conversion.')

# ── Validate ──────────────────────────────────────────────────────────────────
if TRAIN_ZIP is None:
    raise FileNotFoundError(
        'BraTS training data not found.\n'
        'Add the dataset: right panel -> Add Input -> Your Datasets -> brats-men-rt.'
    )

image_ok = any(s in suffix_counts for s in ('t1c','t1ce','t1n','t1','flair'))
if not TRAIN_ZIP.endswith('.zip') and suffix_counts and not image_ok:
    raise FileNotFoundError(
        f'No T1c image files found in {TRAIN_ZIP}\n'
        'Each case folder needs *_t1c.nii or *_t1c.nii.gz + *_gtv.nii or *_gtv.nii.gz'
    )

print('\nDataset found. Ready to proceed.')

In [ ]:
# ── Cell 3: Clone repository from GitHub ─────────────────────────────────────
import os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# Ensure working directory is valid before any git operations
os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

secrets  = UserSecretsClient()
token    = secrets.get_secret('nnunet_kaggle')
repo_url = f'https://{token}@github.com/maryampervaiz-alt/nnunet-training.git'
REPO_DIR = '/kaggle/working/nnunet-training'

subprocess.run(['rm', '-rf', REPO_DIR], capture_output=True)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', repo_url, REPO_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(
        f'git clone failed (rc={result.returncode}):\n{result.stderr}\n'
        'Check that your nnunet_kaggle secret contains a valid GitHub token.'
    )

print('Repository cloned successfully.')
print('Contents:')
for item in sorted(Path(REPO_DIR).iterdir()):
    print(f'  {item.name}/' if item.is_dir() else f'  {item.name}')


In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

def pip_install(*packages: str, quiet: bool = True) -> None:
    """Install one or more packages via pip."""
    args = [sys.executable, '-m', 'pip', 'install'] + list(packages)
    if quiet:
        args.append('-q')
    result = subprocess.run(args)
    if result.returncode != 0:
        raise RuntimeError(f'pip install failed for: {packages}')

print('Installing surface-distance (from GitHub) ...')
pip_install('git+https://github.com/google-deepmind/surface-distance.git')

print('Installing nnunetv2 ...')
pip_install('nnunetv2')

print('Installing remaining dependencies ...')
pip_install(
    'nibabel', 'SimpleITK', 'scipy', 'scikit-image',
    'pandas', 'matplotlib', 'seaborn',
    'loguru', 'rich', 'python-dotenv', 'tqdm', 'mlflow',
)

print('\nAll packages installed.')

In [ ]:
# ── Cell 5: Register custom trainer with nnunetv2 (CRITICAL) ─────────────────
# nnU-Net discovers trainer classes by scanning its own package directory.
# Our custom trainer is outside that package, so we copy it in.

import shutil, importlib.util, os
from pathlib import Path

# Set nnunet env vars NOW so nnunetv2 import doesn't print "not defined" warnings
PROJECT = REPO_DIR  # defined in Cell 3
os.environ.setdefault('nnUNet_raw',          f'{PROJECT}/nnunet_raw')
os.environ.setdefault('nnUNet_preprocessed', f'{PROJECT}/nnunet_preprocessed')
os.environ.setdefault('nnUNet_results',      f'{PROJECT}/nnunet_results')

import nnunetv2

nnunet_pkg_dir = Path(nnunetv2.__file__).parent
trainer_src    = Path(REPO_DIR) / 'src' / 'training' / 'nnunet_trainer_es.py'
trainer_dst    = nnunet_pkg_dir / 'training' / 'nnUNetTrainer' / 'nnUNetTrainerEarlyStopping.py'

if not trainer_src.exists():
    raise FileNotFoundError(f'Custom trainer source not found: {trainer_src}')

shutil.copy2(trainer_src, trainer_dst)
print(f'Registered: {trainer_src.name} → {trainer_dst}')

# Verify the class is importable
spec = importlib.util.spec_from_file_location('nnUNetTrainerEarlyStopping', trainer_dst)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
cls  = getattr(mod, 'nnUNetTrainerEarlyStopping', None)
if cls is None:
    raise ImportError('nnUNetTrainerEarlyStopping not found in copied file')

print(f'Trainer class   : {cls}')
print('Custom trainer registration: OK')

In [ ]:
# ── Cell 6: Configure environment variables ───────────────────────────────────

PROJECT   = REPO_DIR
MAX_CASES = 20     # 20-case subset for quick pilot run

os.environ.update({
    'nnUNet_raw'          : f'{PROJECT}/nnunet_raw',
    'nnUNet_preprocessed' : f'{PROJECT}/nnunet_preprocessed',
    'nnUNet_results'      : f'{PROJECT}/nnunet_results',
    'DATASET_ID'          : '001',
    'DATASET_NAME'        : 'BraTSMENRT',
    'NNUNET_CONFIGURATION': '3d_fullres',
    'NNUNET_SEED'         : '42',
    'ES_PATIENCE'         : '20',
    'ES_MIN_DELTA'        : '0.0001',
    'ES_WARMUP'           : '20',
    'NUM_FOLDS'           : '5',
    'CUDA_VISIBLE_DEVICES': '0',
    'TRAIN_SOURCE'        : str(TRAIN_ZIP),
    'VAL_SOURCE'          : str(VAL_ZIP) if VAL_ZIP else '',
    'LABEL_SUFFIX'        : 'gtv',
    'EXPERIMENT_NAME'     : 'BraTSMENRT_baseline',
    'MLFLOW_TRACKING_URI' : f'{PROJECT}/experiments/mlruns',
    # Disable Python stdout buffering so subprocess output appears immediately.
    'PYTHONUNBUFFERED'    : '1',
    # Limit DA workers to avoid fork-based deadlocks inside Kaggle subprocess chain.
    'nnUNet_n_proc_DA'    : '4',
    # Disable torch.compile to avoid 20-40 min JIT compilation on the first epoch.
    # On the full dataset with many epochs, set this to 'true' for faster training.
    'nnUNet_compile'      : 'false',
})

for d in [
    'nnunet_raw', 'nnunet_preprocessed', 'nnunet_results',
    'metrics', 'results', 'visualizations',
    'inference_outputs', 'logs', 'experiments', 'checkpoints',
]:
    Path(f'{PROJECT}/{d}').mkdir(parents=True, exist_ok=True)

print('Environment configured:')
for k in ['nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results',
          'DATASET_ID', 'DATASET_NAME', 'NNUNET_CONFIGURATION']:
    print(f'  {k:25} = {os.environ[k]}')
print(f'  TRAIN_SOURCE              = {TRAIN_ZIP}')
print(f'  MAX_CASES                 = {MAX_CASES}')
print(f'  torch.compile             = {os.environ.get("nnUNet_compile", "true")}')


In [ ]:
# ── Cell 7: Set working directory to the repo ─────────────────────────────────
# All subsequent ! commands and relative paths resolve from here.

os.chdir(PROJECT)

# Add repo root to Python path so src.* imports work in notebook cells
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print(f'Working directory: {os.getcwd()}')
print(f'Python path includes: {sys.path[0]}')

In [ ]:
# ── Cell 8: STEP 1 — Convert raw BraTS data to nnU-Net format ────────────
# Expected time: 2-4 minutes for 20 cases (training only, validation skipped).

cmd = [
    sys.executable, '-u', 'scripts/01_prepare_dataset.py',
    '--train',    str(TRAIN_ZIP),
    '--skip-val',
    '--log-dir',  'logs',
]
if MAX_CASES is not None:
    cmd += ['--max-cases', str(MAX_CASES)]

cmd_str = ' '.join(cmd)
print(f'Running: {cmd_str}')
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'01_prepare_dataset.py failed (rc={proc.returncode})')
print()
print('Dataset preparation complete.')


In [ ]:
# ── Cell 9: Verify dataset structure ─────────────────────────────────────────
import json
from pathlib import Path

raw_dir     = Path(os.environ['nnUNet_raw'])
dataset_dir = raw_dir / 'Dataset001_BraTSMENRT'
images_tr   = dataset_dir / 'imagesTr'
labels_tr   = dataset_dir / 'labelsTr'
images_ts   = dataset_dir / 'imagesTs'

n_tr  = len(list(images_tr.glob('*_0000.nii.gz')))
n_lbl = len(list(labels_tr.glob('*.nii.gz')))
n_ts  = len(list(images_ts.glob('*_0000.nii.gz'))) if images_ts.exists() else 0

with open(dataset_dir / 'dataset.json') as fh:
    ds = json.load(fh)

print(f'imagesTr images  : {n_tr}')
print(f'labelsTr labels  : {n_lbl}')
print(f'imagesTs images  : {n_ts}')
print(f'numTraining (json): {ds["numTraining"]}')
print(f'channel_names    : {ds["channel_names"]}')
print(f'labels           : {ds["labels"]}')

assert n_tr == n_lbl, f'Image/label count mismatch: {n_tr} vs {n_lbl}'
assert ds['numTraining'] == n_tr, (
    f'numTraining in dataset.json ({ds["numTraining"]}) does not match '
    f'actual file count ({n_tr})'
)
print('\nDataset structure OK.')

In [ ]:
# ── Cell 10: STEP 2 — nnU-Net Planning & Preprocessing ───────────────────────
# Runs nnUNetv2_plan_and_preprocess. Runtime scales with dataset size:
# ~5 min for 20 cases, ~20-40 min for the full dataset.

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/02_preprocess.py', '--log-dir', 'logs'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'02_preprocess.py failed (rc={proc.returncode})')
print('Preprocessing complete.')


In [ ]:
# ── Cell 11: Integrity Check — Validate Dataset Before Training ───────────────
# Checks every case for:
#   - Readable NIfTI files
#   - Matching image/label spatial dimensions
#   - Binary label values {0, 1}
#   - No duplicate case IDs

from src.data.integrity_checker import IntegrityChecker

checker = IntegrityChecker(dataset_dir=dataset_dir, expected_label_values={0, 1})
report  = checker.run()

print('=' * 60)
print('  INTEGRITY CHECK REPORT')
print('=' * 60)
print(report.summary())
print('=' * 60)

failed = [r for r in report.case_reports if not r.ok]
if failed:
    for r in failed:
        print(f'  FAIL [{r.case_id}]: {r.errors}')
    raise AssertionError(
        f'{len(failed)} case(s) failed integrity checks. '
        'Fix the issues above before proceeding.'
    )

print('All cases passed integrity checks. Safe to train.')


In [ ]:
# ── Cell 12: Verify preprocessing + show CV splits ───────────────────────────

preproc_dir = Path(os.environ['nnUNet_preprocessed']) / 'Dataset001_BraTSMENRT'

# List preprocessed files
print('Preprocessed directory contents:')
for item in sorted(preproc_dir.iterdir()):
    print(f'  {item.name}')

# Show splits
splits_file = preproc_dir / 'splits_final.json'
if splits_file.exists():
    with open(splits_file) as fh:
        splits = json.load(fh)
    print(f'\n5-fold cross-validation splits ({len(splits)} folds):')
    for i, fold in enumerate(splits):
        print(f'  Fold {i}: {len(fold["train"])} train | {len(fold["val"])} val')
else:
    print('\nsplits_final.json not found — preprocessing may not have completed.')

# Show nnU-Net plans
for plans_file in sorted(preproc_dir.glob('nnUNetPlans*.json')):
    with open(plans_file) as fh:
        plans = json.load(fh)
    print(f'\nPlans from {plans_file.name}:')
    for cfg_name, cfg in plans.get('configurations', {}).items():
        print(
            f'  [{cfg_name}]  '
            f'patch_size={cfg.get("patch_size")}  '
            f'spacing={cfg.get("spacing")}  '
            f'batch_size={cfg.get("batch_size")}'
        )

In [ ]:
# -- Cell 13: STEP 3 -- Train All 5 Folds (10 epochs pilot) --
# Trains folds 0-4 sequentially. Saves checkpoints after each fold.
# Runtime: ~3-5 min/epoch x 10 epochs x 5 folds approx 2.5-4 hours on T4.
#
# RESUME: Set CONTINUE_TRAINING = True to resume from epoch 10 to 50.
# Already-completed folds are skipped when SKIP_DONE = True.

ALL_FOLDS         = [0, 1, 2, 3, 4]
NUM_EPOCHS        = 10      # change to 50 when resuming
CONTINUE_TRAINING = False   # True to resume from checkpoint_latest.pth
SKIP_DONE         = False   # True to skip folds that already have checkpoint_final.pth

TRAINER = "nnUNetTrainerEarlyStopping"
PLANS   = "nnUNetPlans"
CONFIG  = os.environ["NNUNET_CONFIGURATION"]
SAVE    = "/kaggle/working/outputs"

def _ckpt_dir(fold):
    ds = f'Dataset{os.environ["DATASET_ID"].zfill(3)}_{os.environ["DATASET_NAME"]}'
    return (Path(os.environ["nnUNet_results"]) / ds
            / f"{TRAINER}__{PLANS}__{CONFIG}" / f"fold_{fold}")

def _is_done(fold):
    return (_ckpt_dir(fold) / "checkpoint_final.pth").exists()

def _save_fold_checkpoints(fold):
    src = _ckpt_dir(fold)
    dst = Path(SAVE) / "nnunet_results" / src.relative_to(os.environ["nnUNet_results"])
    dst.mkdir(parents=True, exist_ok=True)
    for f in list(src.glob("*.pth")) + list(src.glob("training_log*.txt")):
        shutil.copy2(f, dst / f.name)
    metrics_src = Path("metrics") / f"fold_{fold}_training.csv"
    if metrics_src.exists():
        md = Path(SAVE) / "metrics"
        md.mkdir(parents=True, exist_ok=True)
        shutil.copy2(metrics_src, md / metrics_src.name)
    n = sum(1 for _ in dst.rglob("*") if Path(_).is_file())
    print(f"  [fold {fold}] Checkpoints saved to outputs ({n} files)")

fold_results = {}

for fold in ALL_FOLDS:
    print()
    print("=" * 64)
    if SKIP_DONE and not CONTINUE_TRAINING and _is_done(fold):
        print(f"  Fold {fold}: SKIPPED (checkpoint_final.pth exists)")
        fold_results[fold] = 0
        continue
    action = "Resuming" if CONTINUE_TRAINING else "Training"
    print(f"  {action} fold {fold} for {NUM_EPOCHS} epochs ...")
    print("=" * 64)

    cmd = [
        sys.executable, "-u", "scripts/03_train.py",
        "--folds",           str(fold),
        "--num-epochs",      str(NUM_EPOCHS),
        "--seed",            "42",
        "--trainer",         TRAINER,
        "--es-patience",     "50",
        "--es-min-delta",    "0.0001",
        "--es-warmup",       "50",
        "--log-dir",         "logs",
        "--metrics-dir",     "metrics",
        "--checkpoints-dir", "checkpoints",
    ]
    if CONTINUE_TRAINING:
        cmd.append("--continue-training")

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    fold_results[fold] = proc.returncode

    if proc.returncode != 0:
        print(f"ERROR: Fold {fold} failed (rc={proc.returncode}). Saving partial checkpoints.")
    else:
        print(f"Fold {fold} complete.")

    _save_fold_checkpoints(fold)

print()
print("=" * 64)
print("All-fold training summary:")
for fold, rc in fold_results.items():
    print(f"  Fold {fold}: {"OK" if rc == 0 else f"FAILED rc={rc}"}")  # noqa: E501
print("=" * 64)


In [ ]:
# -- Cell 13b: Training History -- All Folds --
import pandas as pd
from pathlib import Path

print("Per-fold training summary:")
print("=" * 70)
print(f"  {chr(70):<4}  {chr(69):>6}  {"Best Dice":>10}  {"Final Dice":>11}  {"Final Loss":>11}")
print(f"  {"Fold":>4}  {"Epochs":>6}  {"Best Dice":>10}  {"Final Dice":>11}  {"Final Loss":>11}")
print("  " + "-" * 52)

all_hists = {}
for fold in [0, 1, 2, 3, 4]:
    csv = Path(f"metrics/fold_{fold}_training.csv")
    if not csv.exists():
        print(f"  {fold:>4}  (no training log yet)")
        continue
    hist = pd.read_csv(csv)
    all_hists[fold] = hist
    n_ep       = len(hist)
    best_dice  = hist["val_dice"].max()    if "val_dice"   in hist.columns else float("nan")
    final_dice = hist["val_dice"].iloc[-1]  if "val_dice"   in hist.columns else float("nan")
    final_loss = hist["train_loss"].iloc[-1] if "train_loss" in hist.columns else float("nan")
    print(f"  {fold:>4}  {n_ep:>6}  {best_dice:>10.4f}  {final_dice:>11.4f}  {final_loss:>11.4f}")

print("=" * 70)

for fold, hist in all_hists.items():
    print(f"
Fold {fold} epoch log (every ~5th epoch):")
    cols = [c for c in ["epoch","train_loss","val_loss","val_dice","epoch_time_s"] if c in hist.columns]
    step = max(1, len(hist) // 10)
    print(hist.iloc[::step][cols].round(4).to_string(index=False))


In [ ]:
# -- Cell 14: Verify Checkpoints -- All Folds --
TRAINER = "nnUNetTrainerEarlyStopping"
PLANS   = "nnUNetPlans"
CONFIG  = os.environ["NNUNET_CONFIGURATION"]
DS      = f'Dataset{os.environ["DATASET_ID"].zfill(3)}_{os.environ["DATASET_NAME"]}'
nnunet_results = Path(os.environ["nnUNet_results"])

print("Checkpoint verification:")
print("=" * 70)
all_ok = True
for fold in [0, 1, 2, 3, 4]:
    ckpt_dir = nnunet_results / DS / f"{TRAINER}__{PLANS}__{CONFIG}" / f"fold_{fold}"
    if not ckpt_dir.exists():
        print(f"  Fold {fold}: NOT TRAINED YET")
        continue
    best   = ckpt_dir / "checkpoint_best.pth"
    final  = ckpt_dir / "checkpoint_final.pth"
    latest = ckpt_dir / "checkpoint_latest.pth"
    main   = final if final.exists() else latest
    b_ok   = best.exists()
    m_ok   = main.exists()
    b_sz   = f"{best.stat().st_size/1e6:.1f} MB" if b_ok else "MISSING"
    m_sz   = f"{main.stat().st_size/1e6:.1f} MB" if m_ok else "MISSING"
    status = "OK" if (b_ok and m_ok) else "INCOMPLETE"
    if not (b_ok and m_ok): all_ok = False
    print(f"  Fold {fold} [{status}]  best={b_sz}  {main.name}={m_sz}")
print("=" * 70)
print("All folds OK." if all_ok else "Some folds incomplete -- see above.")


---
## After Training All Folds

Proceed in order:
- **Cell 14** - Verify that all fold checkpoints exist
- **Cell 15** - Run per-fold inference on validation splits
- **Cell 15b** - Post-processing (CC threshold sweep + Wilcoxon test)
- **Cell 16** - Evaluate all folds (DSC, HD95, NSD, Precision, Recall)
- **Cell 17** - Publication-ready visualizations
- **Cell 18** - Full metrics table (all folds combined)
- **Cell 20** - Save all outputs before session ends
- **Cell 21** - Resume training (10 -> 50 epochs per fold)


In [ ]:
# -- Cell 15: STEP 4 -- Inference All Available Folds --
# Runs nnUNetv2_predict on each fold's held-out validation split.
# Output: inference_outputs/cv/fold_{N}/*.nii.gz
# Skips folds that have no checkpoint yet.

import os, subprocess, sys
from pathlib import Path

TRAINER = "nnUNetTrainerEarlyStopping"
PLANS   = "nnUNetPlans"
CONFIG  = os.environ["NNUNET_CONFIGURATION"]
DS      = f'Dataset{os.environ["DATASET_ID"].zfill(3)}_{os.environ["DATASET_NAME"]}'

def _has_checkpoint(fold):
    d = (Path(os.environ["nnUNet_results"]) / DS
         / f"{TRAINER}__{PLANS}__{CONFIG}" / f"fold_{fold}")
    return (d / "checkpoint_final.pth").exists() or (d / "checkpoint_best.pth").exists()

trained_folds = [f for f in [0,1,2,3,4] if _has_checkpoint(f)]
print(f"Trained folds with checkpoints: {trained_folds}")

for fold in trained_folds:
    print(f"
Running inference for fold {fold} ...")
    proc = subprocess.Popen(
        [
            sys.executable, "-u", "scripts/04_inference.py",
            "--cv-mode",
            "--folds",   str(fold),
            "--output",  "inference_outputs/cv",
            "--log-dir", "logs",
        ],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Fold {fold} inference failed (rc={proc.returncode})")
    else:
        preds = list(Path(f"inference_outputs/cv/fold_{fold}").glob("*.nii.gz"))
        print(f"Fold {fold}: {len(preds)} predictions")

print("
Inference complete for all available folds.")


In [ ]:
# -- Cell 15b: Post-processing -- CC Threshold Sweep + Wilcoxon Test --
# Sweeps over candidate min_voxel thresholds and selects the best on the
# VALIDATION set. Reports Wilcoxon signed-rank test (raw vs post-processed).
# WARNING: threshold selected HERE must be reported honestly in the paper as
# "selected on the validation set" -- do not evaluate on a separate test split
# without a fresh threshold selection.

import numpy as np
import nibabel as nib
from scipy import ndimage
import pandas as pd
import subprocess, sys
from pathlib import Path

# -- Threshold candidates to sweep --
THRESHOLDS = [10, 25, 50, 100, 200, 500]

gt_dir = Path(os.environ["nnUNet_raw"]) / "Dataset001_BraTSMENRT" / "labelsTr"

def cc_filter(arr, min_voxels):
    labeled, n = ndimage.label(arr > 0)
    if n == 0:
        return arr.copy().astype(np.uint8)
    sizes = ndimage.sum(arr > 0, labeled, range(1, n + 1))
    keep = np.zeros_like(labeled, dtype=bool)
    for idx, sz in enumerate(sizes, 1):
        if sz >= min_voxels:
            keep |= (labeled == idx)
    return keep.astype(np.uint8)

def mean_dice(pred_dir, gt_dir):
    preds = sorted(Path(pred_dir).glob("*.nii.gz"))
    if not preds:
        return float("nan")
    dices = []
    for pp in preds:
        gp = gt_dir / pp.name
        if not gp.exists():
            continue
        p = np.asarray(nib.load(pp).dataobj, dtype=np.uint8)
        g = np.asarray(nib.load(gp).dataobj, dtype=np.uint8)
        tp = int(np.logical_and(p, g).sum())
        denom = int(p.sum()) + int(g.sum())
        dices.append(2 * tp / denom if denom > 0 else 1.0)
    return float(np.mean(dices)) if dices else float("nan")

# -- Run sweep over all trained folds --
trained_folds = [f for f in [0,1,2,3,4]
                 if Path(f"inference_outputs/cv/fold_{f}").exists()]
print(f"Folds with predictions: {trained_folds}")

sweep_results = []
for thresh in THRESHOLDS:
    fold_dices = []
    for fold in trained_folds:
        src = Path(f"inference_outputs/cv/fold_{fold}")
        dst = Path(f"inference_outputs/cv/fold_{fold}_pp_{thresh}")
        dst.mkdir(parents=True, exist_ok=True)
        for pp in src.glob("*.nii.gz"):
            img = nib.load(pp)
            cleaned = cc_filter(np.asarray(img.dataobj, dtype=np.uint8), thresh)
            nib.save(nib.Nifti1Image(cleaned, img.affine, img.header), dst / pp.name)
        fold_dices.append(mean_dice(dst, gt_dir))
    sweep_results.append({"min_voxels": thresh, "mean_DSC": float(np.nanmean(fold_dices))})

sweep_df = pd.DataFrame(sweep_results)
print("
CC threshold sweep results:")
print(sweep_df.to_string(index=False))

BEST_THRESH = int(sweep_df.loc[sweep_df["mean_DSC"].idxmax(), "min_voxels"])
print(f"
Best threshold: {BEST_THRESH} voxels (highest mean DSC)")

# -- Apply best threshold and save as final post-processed predictions --
for fold in trained_folds:
    src = Path(f"inference_outputs/cv/fold_{fold}")
    dst = Path(f"inference_outputs/cv/fold_{fold}_postproc")
    dst.mkdir(parents=True, exist_ok=True)
    n_changed = 0
    for pp in src.glob("*.nii.gz"):
        img = nib.load(pp)
        arr = np.asarray(img.dataobj, dtype=np.uint8)
        cleaned = cc_filter(arr, BEST_THRESH)
        if not np.array_equal(arr, cleaned):
            n_changed += 1
        nib.save(nib.Nifti1Image(cleaned, img.affine, img.header), dst / pp.name)
    print(f"Fold {fold}: {n_changed}/{len(list(src.glob("*.nii.gz")))} predictions modified")

print(f"
Post-processing complete. Threshold = {BEST_THRESH} voxels.")
print("Note for paper: report this threshold as selected on the validation set.")


In [ ]:
# -- Cell 16: STEP 5 -- Evaluate All Folds + Wilcoxon Significance Test --

import subprocess, sys
from pathlib import Path

# -- CV evaluation (all folds) --
proc = subprocess.Popen(
    [
        sys.executable, "-u", "scripts/05_evaluate.py",
        "--cv-mode",
        "--results-dir",  "results",
        "--show-rankings",
        "--latex",
        "--stat-test",
        "--log-dir",      "logs",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode not in (0, 1):
    print(f"WARNING: evaluate exited with rc={proc.returncode}")

print("
Evaluation complete.")
for f in sorted(Path("results").glob("*.csv")):
    print(f"  {f.name}")


In [ ]:
# -- Cell 17: STEP 6 -- Publication-Ready Visualizations --
# Generates: segmentation overlays (GT/Pred/TP/FP/FN), violin plots,
# box plots, volume scatter, training curves for all folds.

import subprocess, sys
from pathlib import Path

result = subprocess.run(
    [
        sys.executable, "scripts/06_visualize.py",
        "--all",
        "--cv-mode",
        "--results-dir",  "results",
        "--metrics-dir",  "metrics",
        "--output-dir",   "visualizations",
        "--n-cases",      "15",
        "--log-dir",      "logs",
    ],
    env=os.environ.copy(),
)
if result.returncode != 0:
    print(f"Warning: visualize exited rc={result.returncode} (some plots may be missing)")

viz_dir = Path("visualizations")
print("
Visualization files:")
for p in sorted(viz_dir.rglob("*.png")):
    size_kb = p.stat().st_size // 1024
    print(f"  {p.relative_to(viz_dir)}  ({size_kb} KB)")


In [ ]:
# -- Cell 18: Quantitative Results -- All Folds Combined --
import pandas as pd
import numpy as np
from pathlib import Path

_DISPLAY = {
    "dice":                "DSC",
    "hd95":                "HD95 (mm)",
    "hd":                  "HD (mm)",
    "nsd":                 "NSD",
    "precision":           "Precision",
    "recall":              "Recall",
    "specificity":         "Specificity",
    "volume_similarity":   "Volume Similarity",
    "abs_volume_error_ml": "Abs. Volume Error (ml)",
}

def show_table(csv_path, title):
    p = Path(csv_path)
    if not p.exists():
        print(f"{title}: not found ({p})"); return
    df = pd.read_csv(p)
    cols = [c for c in _DISPLAY if c in df.columns]
    if not cols:
        print(f"{title}: no metric columns"); return
    rows = []
    for col in cols:
        v = pd.to_numeric(df[col], errors="coerce").replace([float("inf"), float("-inf")], float("nan")).dropna()
        if v.empty: continue
        rows.append({
            "Metric":       _DISPLAY[col],
            "Mean +/- Std": f"{v.mean():.4f} +/- {v.std():.4f}",
            "Median [IQR]": f"{v.median():.4f} [{v.quantile(.25):.4f}, {v.quantile(.75):.4f}]",
            "Min":          f"{v.min():.4f}",
            "Max":          f"{v.max():.4f}",
        })
    summary = pd.DataFrame(rows).set_index("Metric")
    print(f"
{title} (n={len(df)} cases)")
    print("=" * 90)
    print(summary.to_string())
    print("=" * 90)
    if "dice" in df.columns:
        n_out = int((pd.to_numeric(df["dice"], errors="coerce") < 0.5).sum())
        if n_out:
            print(f"  Outliers (DSC < 0.5): {n_out} cases")

# Per-fold results
for fold in [0, 1, 2, 3, 4]:
    show_table(f"results/fold_{fold}_per_case.csv", f"Fold {fold} Validation")

# Combined CV result
show_table("results/cv_combined.csv", "All Folds Combined (CV)")

# Wilcoxon result (raw vs post-processed)
wil = Path("results/cv_wilcoxon.csv")
if wil.exists():
    print("
Wilcoxon Signed-Rank Test: raw vs post-processed")
    print(pd.read_csv(wil).to_string(index=False))


In [ ]:
# -- Cell 19: Show Visualizations --
from IPython.display import Image, display
from pathlib import Path

overlay_dir = Path("visualizations/overlays")
if overlay_dir.exists():
    for img_path in sorted(overlay_dir.glob("*.png"))[:3]:
        print(f"\n{img_path.name}")
        display(Image(str(img_path), width=900))
else:
    print("No overlays found -- run Cell 17 first")

for plot_name in ["metrics_violin.png", "metrics_boxplot.png",
                  "volume_scatter.png", "fold_comparison.png"]:
    p = Path("visualizations") / plot_name
    if p.exists():
        print(f"\n{plot_name}")
        display(Image(str(p), width=900))

for fold in [0, 1, 2, 3, 4]:
    p = Path(f"visualizations/training_curve_fold_{fold}.png")
    if p.exists():
        print(f"\nTraining curve -- fold {fold}")
        display(Image(str(p), width=800))


In [ ]:
# -- Cell 20: Save All Outputs to /kaggle/working/outputs --
# ALWAYS run this cell before closing the session.
# Kaggle deletes /kaggle/working/nnunet-training on session end.

import shutil
from pathlib import Path

SAVE = "/kaggle/working/outputs"
Path(SAVE).mkdir(parents=True, exist_ok=True)

to_save = [
    ("results",           f"{SAVE}/results"),
    ("metrics",           f"{SAVE}/metrics"),
    ("visualizations",    f"{SAVE}/visualizations"),
    ("inference_outputs", f"{SAVE}/inference_outputs"),
    ("logs",              f"{SAVE}/logs"),
    ("checkpoints",       f"{SAVE}/checkpoints"),
]
# Save native nnU-Net checkpoints separately
to_save.append((os.environ["nnUNet_results"], f"{SAVE}/nnunet_results"))

for src, dst in to_save:
    if Path(src).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        n = sum(1 for _ in Path(dst).rglob("*") if Path(_).is_file())
        print(f"  {n:5d} files  {src} -> {dst}")
    else:
        print(f"  SKIPPED: {src}")

print(f"\nAll outputs saved to {SAVE}")

pth_files = sorted(Path(SAVE).rglob("*.pth"))
if pth_files:
    print(f"\n{len(pth_files)} checkpoint files:")
    for p in pth_files:
        print(f"  {p.relative_to(SAVE)}  ({p.stat().st_size/1e6:.1f} MB)")


---
## Cell 21 -- Resume Training (10 to 50 Epochs per Fold)

After saving outputs (Cell 20) and starting a new Kaggle session:

1. Run Cells 1-7 (setup)
2. Set `CONTINUE_TRAINING = True` and `NUM_EPOCHS = 50` in Cell 21
3. Run Cell 21 -- auto-restores checkpoints and resumes each fold

Folds that already completed `NUM_EPOCHS` are skipped when `SKIP_DONE = True`.


In [ ]:
# -- Cell 21: Resume Training -- 10 to 50 Epochs per Fold --
# Run in a new Kaggle session after restoring saved checkpoints.

ALL_FOLDS         = [0, 1, 2, 3, 4]
NUM_EPOCHS        = 50      # total target epochs
CONTINUE_TRAINING = True    # MUST be True
SKIP_DONE         = True    # skip folds already at NUM_EPOCHS

import shutil
import pandas as pd

TRAINER = "nnUNetTrainerEarlyStopping"
PLANS   = "nnUNetPlans"
CONFIG  = os.environ["NNUNET_CONFIGURATION"]

# Restore checkpoints from previous session outputs
saved_results = Path("/kaggle/working/outputs/nnunet_results")
if saved_results.exists():
    shutil.copytree(str(saved_results), os.environ["nnUNet_results"], dirs_exist_ok=True)
    print(f"Checkpoints restored from {saved_results}")
else:
    print("WARNING: No saved checkpoints found at", saved_results)
    print("Run Cell 20 before closing the previous session.")

saved_metrics = Path("/kaggle/working/outputs/metrics")
if saved_metrics.exists():
    shutil.copytree(str(saved_metrics), "metrics", dirs_exist_ok=True)
    print("Metrics CSVs restored.")

def _ckpt_dir(fold):
    ds = f"Dataset{os.environ[chr(68)+chr(65)+chr(84)+chr(65)+chr(83)+chr(69)+chr(84)+chr(95)+chr(73)+chr(68)].zfill(3)}"
    ds = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
    return (Path(os.environ["nnUNet_results"]) / ds
            / f"{TRAINER}__{PLANS}__{CONFIG}" / f"fold_{fold}")

def _current_epoch(fold):
    csv = Path(f"metrics/fold_{fold}_training.csv")
    if not csv.exists():
        return 0
    return len(pd.read_csv(csv))

def _save_fold_ckpt(fold):
    src = _ckpt_dir(fold)
    dst = Path("/kaggle/working/outputs/nnunet_results") / src.relative_to(os.environ["nnUNet_results"])
    dst.mkdir(parents=True, exist_ok=True)
    for fp in list(src.glob("*.pth")) + list(src.glob("training_log*.txt")):
        shutil.copy2(fp, dst / fp.name)
    mcsv = Path("metrics") / f"fold_{fold}_training.csv"
    if mcsv.exists():
        md = Path("/kaggle/working/outputs/metrics")
        md.mkdir(parents=True, exist_ok=True)
        shutil.copy2(mcsv, md / mcsv.name)
    print(f"  [fold {fold}] Checkpoints saved to outputs")

fold_results = {}
for fold in ALL_FOLDS:
    print()
    print("=" * 64)
    cur_ep = _current_epoch(fold)
    if SKIP_DONE and cur_ep >= NUM_EPOCHS:
        print(f"  Fold {fold}: SKIPPED ({cur_ep}/{NUM_EPOCHS} epochs done)")
        fold_results[fold] = 0
        continue
    print(f"  Resuming fold {fold}: {cur_ep} -> {NUM_EPOCHS} epochs")
    print("=" * 64)

    cmd = [
        sys.executable, "-u", "scripts/03_train.py",
        "--folds",           str(fold),
        "--num-epochs",      str(NUM_EPOCHS),
        "--seed",            "42",
        "--trainer",         TRAINER,
        "--es-patience",     "50",
        "--es-min-delta",    "0.0001",
        "--es-warmup",       "50",
        "--log-dir",         "logs",
        "--metrics-dir",     "metrics",
        "--checkpoints-dir", "checkpoints",
        "--continue-training",
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    fold_results[fold] = proc.returncode

    ok = proc.returncode == 0
    print(f"\nFold {fold}: {"OK" if ok else f"FAILED rc={proc.returncode}"}")
    _save_fold_ckpt(fold)

print()
print("=" * 64)
print("Resume summary:")
for fold, rc in fold_results.items():
    ep = _current_epoch(fold)
    s = "OK" if rc == 0 else f"FAILED rc={rc}"
    print(f"  Fold {fold}: {ep} epochs | {s}")
print("=" * 64)
print("\nNext: re-run Cells 15-18 for 50-epoch evaluation and visualizations.")
